> 📒 이 노트북은 [펭귄 데이터로 배우는 인공지능](https://yanghyeokjin33.github.io/Algorithm/) 수업에서 쓰는 실습 노트북이에요. 원본: [https://github.com/YANGHYEOKJIN33/Algorithm](https://github.com/YANGHYEOKJIN33/Algorithm)

# 🕸 웹 크롤링으로 펭귄 데이터 모으기

**목표**: 웹 페이지의 표를 프로그램으로 읽어 와서(크롤링) 데이터프레임으로 만들고, CSV 파일로 저장한다.

오늘 할 일은 사이트에서 배운 의사코드와 똑같아요.

    행목록 ← 빈 리스트
    반복: 쪽 번호 p = 1부터 7까지
        HTML ← p쪽 주소에 요청해서 받아 온다
        문서 ← HTML을 분석해 태그 나무로 만든다
        줄들 ← 문서에서 <tr> 태그를 모두 찾는다
        반복: 각 줄 tr (제목 줄은 건너뜀)
            행목록에 tr 안 <td> 글자들을 추가한다
    표 ← 행목록으로 데이터프레임을 만든다

연습 사이트: https://yanghyeokjin33.github.io/Algorithm/practice/ (크롤링 연습용으로 만든 사이트라서 마음 놓고 연습해도 돼요)

> **Colab 사용법** — 셀 왼쪽의 ▶ 단추를 누르거나 **Shift + Enter**를 누르면 그 셀이 실행돼요.
> 맨 위 셀부터 **차례로** 실행하세요. 코드를 고쳐 보고 싶다면 먼저 **파일 → 드라이브에 사본 저장**을 눌러 내 사본을 만들어 두세요.
> 파이썬을 몰라도 괜찮아요. 코드 위의 설명을 사이트의 의사코드와 한 줄씩 맞춰 읽으면 돼요.

- 크롤링에 쓸 도구 네 가지를 불러와요(import).
- SITE에 연습 사이트 주소를 넣어 둬요. 다른 사이트를 크롤링하고 싶으면 이 주소만 바꾸면 돼요.

In [ ]:
import requests                      # 웹 페이지 요청하기
from bs4 import BeautifulSoup        # HTML 분석하기
import pandas as pd                  # 표(데이터프레임) 다루기
import time                          # 잠깐 쉬기 (서버 예절)

SITE = 'https://yanghyeokjin33.github.io/Algorithm/practice'
print('준비 완료!')

## 1. 먼저 한 쪽만 — 요청해서 HTML 받아 오기

`requests.get(주소)`는 브라우저처럼 서버에 "이 쪽을 보내 주세요" 하고 **요청**해요. 서버가 돌려준 **응답**에서 `.text`를 꺼내면 HTML 글자가 나와요.

- res에는 서버가 보낸 응답이 담겨요.
- 상태 코드 200은 "잘 받았어요", 404는 "그런 쪽이 없어요"라는 뜻이에요.
- 받은 HTML의 앞부분을 출력해서 <table>, <tr>, <td> 태그가 있는지 눈으로 확인해요.

In [ ]:
res = requests.get(SITE + '/page1.html')
res.encoding = 'utf-8'               # 한글이 깨지지 않게
print('상태 코드:', res.status_code)  # 200이면 성공
print(res.text[:600])                # 앞부분 600글자만 보기

## 2. HTML을 분석해 <tr> 찾기

BeautifulSoup은 길게 이어진 HTML 글자를 **태그 나무**로 정리해 줘요. `find_all('tr')`을 쓰면 표의 줄(<tr>)을 모두 찾아 리스트로 돌려줘요.

- soup에는 분석을 마친 태그 나무가 담겨요.
- trs는 <tr>을 모두 담은 리스트예요. 첫 번째(0번) 줄은 <th>로 된 제목 줄이에요.

In [ ]:
soup = BeautifulSoup(res.text, 'html.parser')
trs = soup.find_all('tr')
print('찾은 <tr> 개수:', len(trs))
print('첫 번째 줄(제목):', trs[0])
print('두 번째 줄(펭귄):', trs[1])

## 3. 칸의 글자 꺼내기 → 행목록에 모으기

제목 줄의 `<th>` 글자는 열 이름으로 쓰고, 나머지 줄의 `<td>` 글자는 행목록에 모아요.

- [td.text for td in …]는 "<td>마다 글자만 꺼내서 리스트로 모아라"라는 뜻이에요.
- rows는 리스트 안에 리스트가 들어 있는 모양이에요. 사이트에서 본 "행목록"과 같아요.

In [ ]:
header = [th.text for th in trs[0].find_all('th')]
rows = []
for tr in trs[1:]:                                  # 제목 줄(0번)은 건너뛰기
    cells = [td.text for td in tr.find_all('td')]   # <td> 안의 글자만
    rows.append(cells)                              # 행목록 맨 뒤에 추가
print('열 이름:', header)
print('모은 줄:', len(rows))
print(rows[:3])

## 4. 여러 쪽 모두 크롤링하기

같은 일을 1쪽부터 7쪽까지 되풀이해요. 한 쪽을 가져올 때마다 `time.sleep(1)`로 1초씩 쉬어서 **서버에 부담을 주지 않도록** 해요. 크롤링할 때 지켜야 할 예절이지요.

- def는 "이런 일을 하는 함수를 만든다"는 뜻이에요. 한 쪽을 크롤링하는 과정을 함수 하나로 묶어 두고, 쪽마다 불러서 써요.
- range(1, 8)은 1부터 7까지예요. 끝 숫자 8은 들어가지 않아요.
- extend는 이번 쪽에서 모은 줄들을 행목록 뒤에 한꺼번에 이어 붙여요.

In [ ]:
def crawl_page(p):
    """p쪽의 표를 크롤링해 (열 이름, 줄 목록)을 돌려준다"""
    res = requests.get(f'{SITE}/page{p}.html')
    res.encoding = 'utf-8'
    soup = BeautifulSoup(res.text, 'html.parser')
    trs = soup.find_all('tr')
    header = [th.text for th in trs[0].find_all('th')]
    rows = [[td.text for td in tr.find_all('td')] for tr in trs[1:]]
    return header, rows

all_rows = []
for p in range(1, 8):              # 1, 2, …, 7쪽
    header, rows = crawl_page(p)
    all_rows.extend(rows)          # 행목록에 이어 붙이기
    print(f'{p}쪽: {len(rows)}줄 → 지금까지 {len(all_rows)}줄')
    time.sleep(1)                  # 예절: 1초 쉬기

## 5. 표로 만들고 CSV 파일로 저장하기

크롤링한 값은 '39.1'처럼 모두 **글자**예요. CSV로 저장했다가 다시 읽으면, 판다스가 숫자는 숫자로, **빈칸은 NaN**으로 알아서 바꿔 줘요.

- DataFrame은 행목록을 행과 열이 있는 표로 바꿔요.
- to_csv로 파일에 저장해요. index=False는 왼쪽의 인덱스 번호는 저장하지 말라는 뜻이에요.
- df.shape가 (345, 9)면 성공이에요! 펭귄은 344마리인데 345줄이 된 까닭은, 사이트 0-4에서 본 것처럼 50번 줄이 두 번 들어갔기 때문이에요. 겹친 줄은 3단원에서 지워요.

In [ ]:
df = pd.DataFrame(all_rows, columns=header)
df.to_csv('penguins.csv', index=False, encoding='utf-8-sig')
df = pd.read_csv('penguins.csv')     # 다시 읽으면 숫자·빈칸이 제대로!
print(df.shape)                       # (행 수, 열 수)
df.head()

- Colab의 파일 내려받기 기능으로 penguins.csv를 내 컴퓨터에 저장해요. 다음 수업에서 이 파일을 다시 써도 돼요.

In [ ]:
# 내 컴퓨터로 내려받기 (Colab에서만 동작)
try:
    from google.colab import files
    files.download('penguins.csv')
except ImportError:
    print('Colab이 아니라서 내려받기를 건너뛰어요. penguins.csv는 지금 폴더에 저장되어 있어요.')

## 🧩 확인해 봐요

1. `df.shape`는 무엇을 알려 주나요?
2. 344마리인데 345줄이 된 까닭은? (힌트: 1쪽 마지막 줄과 2쪽 첫 줄)
3. `df.isnull().sum()`을 실행해 보세요. 비어 있는 칸이 있나요? 자세한 내용은 다음 수업 🔍 가공에서 다뤄요.

## 🚀 도전 — 다른 연습 사이트

[books.toscrape.com](https://books.toscrape.com)은 누구나 크롤링을 연습할 수 있도록 공개된 가상 서점이에요. 책 제목과 가격을 모아 보세요.

- 이 사이트에서는 책 한 권의 정보가 <article class="product_pod"> 안에 들어 있어요. 사이트마다 태그 구조가 다르니, 먼저 페이지 소스(Ctrl+U)를 열어 보고 어떤 태그를 찾을지 정해요.

In [ ]:
try:
    res = requests.get('https://books.toscrape.com/', timeout=10)
    res.encoding = 'utf-8'                           # £ 기호가 깨지지 않게
    soup = BeautifulSoup(res.text, 'html.parser')
    books = []
    for art in soup.find_all('article', class_='product_pod'):
        title = art.h3.a['title']                        # 책 제목
        price = art.find('p', class_='price_color').text  # 가격
        books.append([title, price])
    print(pd.DataFrame(books, columns=['제목', '가격']).head())
except Exception as e:
    print('지금은 연결할 수 없어요:', e)